In [1]:
%pip install pandas rapidfuzz

import pandas as pd
from rapidfuzz import fuzz, process
import sys
# Cell 3: Define helper functions
def get_first_two_words(text):
    """Extract first 2 words from Arabic/English text (handles short titles)"""
    if pd.isna(text) or not isinstance(text, str):
        return ""
    words = text.strip().split()
    return " ".join(words[:2]) if len(words) >= 2 else text.strip()

def merge_csvs(source_path, target_path, output_path, threshold=80):
    # Read CSVs with UTF-8-SIG encoding (handles Arabic + BOM)
    source_df = pd.read_csv(source_path, encoding='utf-8-sig')
    target_df = pd.read_csv(target_path, encoding='utf-8-sig')
    
    # Ensure required columns exist in source
    required_source_cols = ['Title', 'Singer', 'Poem', 'Date', 'YouTube', 'DetailURL']
    for col in required_source_cols:
        if col not in source_df.columns:
            raise ValueError(f"Source CSV missing required column: '{col}'")
    
    # Add new columns to target if missing (initialize as empty)
    new_cols = ['Poem_line_raw', 'Singer', 'Date', 'YouTube', 'DetailURL']
    for col in new_cols:
        if col not in target_df.columns:
            target_df[col] = pd.NA
    
    # Track matched target indices to prevent duplicates
    matched_target_indices = set()
    unmatched_source_rows = []
    
    # Create a mapping from index to title for matching
    available_targets = {
        i: str(target_df.iloc[i]['Title_raw']) 
        for i in range(len(target_df)) 
        if i not in matched_target_indices
    }
    
    # Process each source row
    for idx, row in source_df.iterrows():
        source_title = str(row['Title']).strip()
        prefix = get_first_two_words(source_title)
        
        # Skip empty prefixes
        if not prefix:
            unmatched_source_rows.append(row)
            continue
        
        # Refresh available targets (excluding already matched ones)
        available_targets = {
            i: str(target_df.iloc[i]['Title_raw']) 
            for i in range(len(target_df)) 
            if i not in matched_target_indices
        }
        
        if not available_targets:
            unmatched_source_rows.append(row)
            continue
        
        # Fuzzy match using Levenshtein ratio (prefix vs full target title)
        match_result = process.extractOne(
            prefix,
            available_targets,
            scorer=fuzz.ratio,
            score_cutoff=threshold
        )
        
        if match_result:
            matched_title, score, target_idx = match_result
            
            # EXACT MATCH: Only update Date/YouTube/DetailURL/Singer (don't overwrite poem)
            if target_df.at[target_idx, 'Title_raw'].strip() == source_title.strip():
                print(f"Exact match found for '{source_title}' - updating metadata only")
                target_df.at[target_idx, 'Singer'] = row['Singer']
                target_df.at[target_idx, 'Date'] = row['Date']
                target_df.at[target_idx, 'YouTube'] = row['YouTube']
                target_df.at[target_idx, 'DetailURL'] = row['DetailURL']
            # FUZZY MATCH: Update all fields including poem (since titles are different)
            else:
                print(f"Fuzzy match found for '{source_title}' -> '{target_df.at[target_idx, 'Title_raw']}' - updating all fields")
                target_df.at[target_idx, 'Poem_line_raw'] = row['Poem']
                target_df.at[target_idx, 'Singer'] = row['Singer']
                target_df.at[target_idx, 'Date'] = row['Date']
                target_df.at[target_idx, 'YouTube'] = row['YouTube']
                target_df.at[target_idx, 'DetailURL'] = row['DetailURL']
            
            matched_target_indices.add(target_idx)
        else:
            unmatched_source_rows.append(row)
    
    # Append unmatched source rows as NEW entries
    if unmatched_source_rows:
        new_rows = []
        for row in unmatched_source_rows:
            new_row = {col: pd.NA for col in target_df.columns}
            new_row['Title_raw'] = row['Title']  # Map Title → Title_raw for new rows
            new_row['Singer'] = row['Singer']
            new_row['Poem_line_raw'] = row['Poem']
            new_row['Date'] = row['Date']
            new_row['YouTube'] = row['YouTube']
            new_row['DetailURL'] = row['DetailURL']
            new_rows.append(new_row)
        
        # Append to target DataFrame
        target_df = pd.concat([target_df, pd.DataFrame(new_rows)], ignore_index=True)
    
    # Save result
    target_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n✓ Merged {len(matched_target_indices)} rows")
    print(f"✓ Added {len(unmatched_source_rows)} new rows from source")
    print(f"✓ Output saved to: {output_path}")


# Cell 4: Set your file paths here
input_csv = ".\CSV\Diwan-Hamdan-WIP - wneen_complete.csv"      # Replace with your source CSV filename
target_csv = ".\CSV\Diwan-Hamdan-WIP - Full_poems.csv"     # Replace with your target CSV filename
output_csv = ".\CSV\merged_output.csv"  # Replace with desired output filename

# Run the merge function
merge_csvs(input_csv, target_csv, output_csv)

Note: you may need to restart the kernel to use updated packages.
Exact match found for 'دمع العظيم' - updating metadata only
Fuzzy match found for 'جسر الصداقة' -> 'جسـر الصـداقة' - updating all fields
Exact match found for 'فخر الاجيال' - updating metadata only
Fuzzy match found for 'برج عاجي ( في حضورك )' -> 'بـرج عـاجـي' - updating all fields
Fuzzy match found for 'الصاحب المعني' -> 'الصاحب الـمعني' - updating all fields
Fuzzy match found for 'ترنيمة' -> 'ترنيمــة' - updating all fields
Fuzzy match found for 'طيب القلب' -> 'طيّـب القلـب' - updating all fields
Fuzzy match found for 'انا انا' -> 'انت وانا' - updating all fields
Fuzzy match found for 'فنون الغدر' -> 'فنـون الغـدر' - updating all fields
Fuzzy match found for 'دموع المحبين' -> 'دمـوع الـمـحـبّـين' - updating all fields
Fuzzy match found for 'خلاص يعني' -> 'خـلاص يعـني' - updating all fields
Fuzzy match found for 'انا والبواخر' -> 'انا والبواخر ..' - updating all fields
Exact match found for 'رابع يوم' - updating metadat